# NISAR sobre el Salar del Hombre Muerto: análisis automático

**Cómo usarlo:** en Colab, menú **Entorno de ejecución > Ejecutar todo**. Lo único que te pide es el usuario y la
contraseña de Earthdata (la contraseña no queda guardada). Después corre solo.

Qué hace, en orden:

1. Busca las escenas NISAR GCOV sobre el salar y elige la órbita que cubre la caja (y, entre esas, la de más fechas).
2. Baja **N_FECHAS** fechas repartidas en el período, **de a una**: baja (~2 GB), recorta el salar, guarda el recorte y borra el archivo grande.
3. Limpia los datos: máscara de píxeles válidos y filtro del moteado del radar.
4. **Agrupamiento automático** (k-means) y cálculo del **centro del salar**.
5. **Cambio fecha a fecha** contra la primera fecha: km² que se aclararon o se oscurecieron, con gráfico (capa principal).
6. **Comparación contra el promedio** de todas las fechas (por si la primera fue anómala) y **cambio persistente**.
7. **Zonas de cambio** entre la primera y la última fecha: tabla con área y ubicación, y polígonos para Google Earth.
8. **Bloque de piletas:** promedio de HH y HV por fecha y superficie con umbral fino.
9. Capa **provisoria** de superficie muy oscura (posible agua), para revisar con imagen óptica.
10. **Deslizador de fechas** y **GIF animado**.
11. Exporta todo (PNG, GIF, CSV, GeoTIFF, GeoJSON y KMZ) en un **ZIP** que se descarga al final.

Solo usa datos públicos y librerías abiertas. Los resultados describen cambios en la superficie; no atribuyen causas.

In [ ]:
# 1) Instalar lo que falta (Colab ya trae numpy, pandas, matplotlib, scipy, scikit-learn, h5py e imageio)
%pip install -q asf_search rasterio

In [ ]:
import os, time, shutil, zipfile, warnings, json
from datetime import datetime
from getpass import getpass
from pathlib import Path

import asf_search as asf
import h5py
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin
from pyproj import Transformer
from scipy.ndimage import uniform_filter
from scipy import ndimage as ndi
from rasterio import features as rio_features
from shapely.geometry import shape, mapping
from shapely import wkt as shapely_wkt
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore", category=RuntimeWarning)
print("asf_search", asf.__version__)

## 2) Parámetros (se pueden tocar; si no, quedan los de abajo)

In [ ]:
BBOX = (-67.35, -25.70, -66.75, -25.15)   # caja del salar completo (versión de Matías): lon_min, lat_min, lon_max, lat_max

# Puntos de referencia (lon, lat), aproximados a partir de informes públicos: solo para orientarse.
# Nombres geográficos, según las reglas de vocabulario del proyecto.
SITIOS = {
    # El centro del salar se calcula solo en la sección 7 (el punto aproximado anterior caía ~7 km al este).
    "Sector este-sureste": (-66.9124, -25.4094),
    "Sector sur": (-67.11, -25.48),
    "Vega de Los Patos": (-66.954, -25.343),
    "Humedal sector suroeste": (-67.082, -25.473),
}
CAJA_PILETAS = (-66.898, -25.437, -66.868, -25.417)   # bloque de piletas: lon_min, lat_min, lon_max, lat_max
UMBRAL_PILETAS_DB = 2.0   # umbral más fino dentro del bloque de piletas
PERSIST_MIN = 3           # "cambio persistente": mismo sentido en al menos esta cantidad de fechas (de las posteriores a la primera)
UMBRAL_PERSIST_DB = 1.5   # umbral del cambio persistente: más bajo que UMBRAL_CAMBIO_DB porque repetirse en varias fechas ya filtra el ruido
START = datetime(2026, 6, 1)
END = datetime.now()

N_FECHAS = 5            # cuántas fechas bajar (repartidas en el período). 0 = todas las de la órbita
FECHAS_FIJAS = ["2026-06-10", "2026-07-04", "2026-08-21", "2026-09-26", "2026-10-20"]  # fechas de la presentación. None = elegir N_FECHAS repartidas
# Ojo: sin FECHAS_FIJAS, cada pasada nueva de NISAR cambia las fechas elegidas y todos los resultados.
POL = "HHHH"            # polarización principal (la otra es HVHV)
UMBRAL_CAMBIO_DB = 3.0  # cambio mínimo (dB) para contar una zona como "se aclaró" o "se oscureció"
AREA_MIN_KM2 = 0.1      # zonas más chicas que esto se descartan (ruido)
UMBRAL_AGUA_DB = -24.0  # capa secundaria PROVISORIA: HH más oscuro que esto = muy oscuro (posible agua). Con -18 marcaba terreno seco
FILTRO_PX = 5           # ventana del filtro del moteado (5 x 5 píxeles ~ 100 m)
K_GRUPOS = 4            # cantidad de grupos del agrupamiento automático
EXPORTAR_HH = False     # True = también exporta la imagen HH de cada fecha (el ZIP pesa mucho más)
EXPORTAR_CAMBIO_FECHAS = True  # exporta el cambio (dB) de cada fecha contra la primera: cambio_HH_dB_AAAA-MM-DD.tif (para la aplicación)
COBERTURA_MIN = 0.9     # la órbita elegida tiene que cubrir al menos este porcentaje de la caja
ORBITA_FRAME = None     # para forzar una órbita: por ejemplo (118, 165). None = elección automática

DATA_DIR = Path("/content/nisar_tmp") if Path("/content").exists() else Path.home() / "nisar_tmp"
RECORTES = DATA_DIR / "recortes"
SALIDA = DATA_DIR / "salida"
for d in (DATA_DIR, RECORTES, SALIDA):
    d.mkdir(parents=True, exist_ok=True)
GRID = "/science/LSAR/GCOV/grids/frequencyA"

lon0, lat0, lon1, lat1 = BBOX
AOI_WKT = f"POLYGON(({lon0} {lat0},{lon1} {lat0},{lon1} {lat1},{lon0} {lat1},{lon0} {lat0}))"
print("Carpeta de trabajo:", DATA_DIR)

## 3) Login en Earthdata (único paso manual)

In [ ]:
session = asf.ASFSession()
if (Path.home() / ".netrc").exists():
    print("Uso las credenciales guardadas en .netrc")
else:
    session.auth_with_creds(input("Usuario Earthdata: ").strip(), getpass("Contraseña Earthdata: "))
    print("Login OK")

## 4) Buscar escenas y elegir fechas

In [ ]:
opts = asf.ASFSearchOptions(maxResults=500, intersectsWith=AOI_WKT, start=START, end=END,
                            processingLevel=["GCOV"], dataset=["NISAR"], productionConfiguration=["PR"])
response = asf.search(opts=opts)
print("Escenas GCOV encontradas:", len(response))

AOI = shapely_wkt.loads(AOI_WKT)
def cobertura(r):
    # Fracción de la caja que cae dentro de la huella de la escena (0 a 1)
    try:
        return shape(r.geometry).intersection(AOI).area / AOI.area
    except Exception:
        return np.nan

esc = pd.DataFrame([{
    "escena": r.properties.get("sceneName") or r.properties.get("fileID"),
    "fecha": pd.to_datetime(r.properties.get("startTime")),
    "orbita": r.properties.get("pathNumber"),
    "frame": r.properties.get("frameNumber"),
    "direccion": r.properties.get("flightDirection"),
    "cobertura": cobertura(r),
    "producto": r,
} for r in response]).sort_values("fecha").reset_index(drop=True)

# Para cada órbita/frame: cuántas fechas tiene y qué parte de la caja cubre su escena
resumen = (esc.groupby(["orbita", "frame", "direccion"]).agg(n=("escena", "count"), cobertura=("cobertura", "median"),
           desde=("fecha", "min"), hasta=("fecha", "max")))
resumen["cobertura"] = resumen.cobertura.round(2)
cubren = resumen[resumen.cobertura >= COBERTURA_MIN]
if cubren.empty:
    print(f"Ninguna órbita cubre el {COBERTURA_MIN:.0%} de la caja; uso la que más cubre")
    cubren = resumen[resumen.cobertura == resumen.cobertura.max()]
resumen = resumen.sort_values(["cobertura", "n"], ascending=False)
display(resumen)

if ORBITA_FRAME:
    orbita, frame = ORBITA_FRAME
    direccion = resumen.reset_index().query("orbita == @orbita and frame == @frame").direccion.iloc[0]
else:
    orbita, frame, direccion = cubren.sort_values(["n", "cobertura"], ascending=False).index[0]
serie = esc[(esc.orbita == orbita) & (esc.frame == frame)].drop_duplicates("fecha").reset_index(drop=True)
if FECHAS_FIJAS:
    dia = serie.fecha.dt.strftime("%Y-%m-%d")
    faltan = sorted(set(FECHAS_FIJAS) - set(dia))
    if faltan:
        print("ATENCIÓN: estas fechas no están en la órbita elegida:", faltan)
    serie = serie[dia.isin(FECHAS_FIJAS)].reset_index(drop=True)
elif N_FECHAS and len(serie) > N_FECHAS:
    idx = np.unique(np.round(np.linspace(0, len(serie) - 1, N_FECHAS)).astype(int))
    serie = serie.iloc[idx].reset_index(drop=True)
print(f"Órbita {orbita}, frame {frame}, {direccion}: cubre el {resumen.loc[(orbita, frame, direccion), 'cobertura']:.0%} de la caja | uso {len(serie)} fechas")
display(serie[["fecha", "escena"]])

## 5) Bajar, recortar y borrar (de a una escena)

In [ ]:
def epsg_de(g):
    p = g["projection"]
    if "epsg_code" in p.attrs:
        return int(np.array(p.attrs["epsg_code"]).item())
    return int(np.array(p[()]).item())

def recortar(path, bbox=BBOX):
    # Lee solo la ventana del salar: HH, HV y la máscara, con sus coordenadas.
    with h5py.File(path, "r") as f:
        g = f[GRID]
        x, y, epsg = g["xCoordinates"][:], g["yCoordinates"][:], epsg_de(g)
        xs, ys = Transformer.from_crs(4326, epsg, always_xy=True).transform(
            [bbox[0], bbox[2], bbox[0], bbox[2]], [bbox[1], bbox[1], bbox[3], bbox[3]])
        ix = np.where((x >= min(xs)) & (x <= max(xs)))[0]
        iy = np.where((y >= min(ys)) & (y <= max(ys)))[0]
        if ix.size == 0 or iy.size == 0:
            raise ValueError("La caja no cae dentro de " + Path(path).name)
        sx, sy = slice(ix.min(), ix.max() + 1), slice(iy.min(), iy.max() + 1)
        capas = {k: g[k][sy, sx].astype("float32") for k in ("HHHH", "HVHV") if k in g}
        mask = g["mask"][sy, sx] if "mask" in g else np.ones_like(next(iter(capas.values())), dtype="uint8")
    return dict(capas, mask=mask, x=x[sx], y=y[sy], epsg=epsg)

def bajar_y_recortar(fila):
    destino = RECORTES / (fila.escena + ".npz")
    if destino.exists():
        print("  ya estaba:", fila.fecha.date())
        return destino
    urls = [u for u in fila.producto.find_urls(extension=".h5", pattern=r"^(?!.*QA_STATS).*")]
    t0 = time.time()
    asf.download_urls(urls, str(DATA_DIR), session=session)
    h5 = DATA_DIR / Path(urls[0]).name
    r = recortar(h5)
    np.savez_compressed(destino, **r)
    h5.unlink()                                   # se borra el archivo grande
    print(f"  {fila.fecha.date()}: bajado y recortado en {time.time() - t0:.0f} s")
    return destino

for _, fila in serie.iterrows():
    for intento in range(3):
        try:
            bajar_y_recortar(fila)
            break
        except Exception as e:
            print(f"  {fila.fecha.date()}: falló ({str(e)[:120]}); reintento {intento + 1}/3")
            time.sleep(10)
print("Recortes listos:", len(list(RECORTES.glob("*.npz"))))

## 6) Cargar, alinear, enmascarar y filtrar

In [ ]:
def a_db(p):
    return 10 * np.log10(np.where(p > 0, p, np.nan))

def filtrar(p, n=FILTRO_PX):
    # Promedio n x n en potencia lineal, ignorando los píxeles sin dato.
    ok = np.isfinite(p)
    s = uniform_filter(np.where(ok, p, 0), n)
    c = uniform_filter(ok.astype("float32"), n)
    return np.where(ok & (c > 0), s / np.maximum(c, 1e-6), np.nan)

nombres = {f.escena: f.fecha for f in serie.itertuples()}
datos = sorted([(nombres.get(p.stem, pd.NaT), dict(np.load(p))) for p in RECORTES.glob("*.npz") if p.stem in nombres],
               key=lambda t: t[0])
fechas = [d[0] for d in datos]

# Grilla común a todas las fechas
xc = datos[0][1]["x"]; yc = datos[0][1]["y"]
for _, d in datos[1:]:
    xc = np.intersect1d(np.round(xc, 2), np.round(d["x"], 2)); yc = np.intersect1d(np.round(yc, 2), np.round(d["y"], 2))
yc = yc[::-1] if datos[0][1]["y"][0] > datos[0][1]["y"][-1] else yc
EPSG = int(datos[0][1]["epsg"])

def a_grilla(d, capa):
    return d[capa][np.ix_(np.isin(np.round(d["y"], 2), yc), np.isin(np.round(d["x"], 2), xc))]

# Máscara: se muestran sus valores. Válidos = distintos de 0 y de 255; si eso descarta casi todo, se ignora.
m0 = a_grilla(datos[0][1], "mask")
print("Valores de la máscara (primera fecha):", np.unique(m0)[:20])
def validos(d, pot):
    m = a_grilla(d, "mask")
    v = (m > 0) & (m < 255) & np.isfinite(pot) & (pot > 0)
    if v.sum() < 0.2 * (np.isfinite(pot) & (pot > 0)).sum():
        print("  aviso: la máscara descartaba casi todo; uso solo los píxeles con dato")
        v = np.isfinite(pot) & (pot > 0)
    return v

HH, HV = [], []
for _, d in datos:
    hh = a_grilla(d, "HHHH"); v = validos(d, hh)
    HH.append(a_db(filtrar(np.where(v, hh, np.nan))))
    hv = a_grilla(d, "HVHV") if "HVHV" in d else np.full_like(hh, np.nan)
    HV.append(a_db(filtrar(np.where(v, hv, np.nan))))
HH, HV = np.stack(HH), np.stack(HV)
PIX_KM2 = abs(np.diff(xc[:2])[0] * np.diff(yc[:2])[0]) / 1e6
print(f"{len(fechas)} fechas | grilla {HH.shape[1]} x {HH.shape[2]} | píxel {PIX_KM2*1e6:.0f} m² | EPSG {EPSG}")
EXT = [xc.min(), xc.max(), yc.min(), yc.max()]

def marcar_sitios(ax, chico=False):
    # Cruces amarillas con el nombre de cada punto de referencia
    t = Transformer.from_crs(4326, EPSG, always_xy=True)
    for nombre, (lon, lat) in SITIOS.items():
        px, py = t.transform(lon, lat)
        ax.plot(px, py, marker="+", color="yellow", markersize=8 if chico else 12, markeredgewidth=2)
        if not chico:
            ax.annotate(nombre, (px, py), color="yellow", fontsize=8, xytext=(6, 6), textcoords="offset points")
etq = [f.strftime("%Y-%m-%d") for f in fechas]

## 7) Agrupamiento automático (k-means) y centro del salar

Agrupa los píxeles de la última fecha según HH y HV, sin ejemplos marcados por nadie, ordenados del más oscuro (1) al
más claro. En la prueba real, el grupo más claro separó el núcleo del salar casi perfecto. Con ese grupo se calcula el
**centro del salar**, que sirve de referencia para ubicar las zonas de cambio. **No es un modelo entrenado ni validado.**

In [ ]:
X = np.stack([HH[-1], HV[-1]], -1).reshape(-1, 2)
ok = np.all(np.isfinite(X), 1)
if ok.sum() < K_GRUPOS * 100:          # sin HV válido, se agrupa solo con HH
    X = HH[-1].reshape(-1, 1); ok = np.isfinite(X[:, 0])
rng = np.random.default_rng(0)
muestra = X[ok][rng.choice(ok.sum(), min(200_000, ok.sum()), replace=False)]
km = KMeans(n_clusters=K_GRUPOS, n_init=5, random_state=0).fit(muestra)
orden = np.argsort(km.cluster_centers_[:, 0])
rango = np.empty(K_GRUPOS, int); rango[orden] = np.arange(K_GRUPOS)
GRUPOS = np.full(ok.shape, np.nan); GRUPOS[ok] = rango[km.predict(X[ok])] + 1
GRUPOS = GRUPOS.reshape(HH[-1].shape)

tabla = pd.DataFrame({"grupo": np.arange(1, K_GRUPOS + 1),
                      "HH_medio_dB": km.cluster_centers_[orden, 0].round(1),
                      "area_km2": [round((GRUPOS == g).sum() * PIX_KM2, 2) for g in range(1, K_GRUPOS + 1)]})
display(tabla); tabla.to_csv(SALIDA / "grupos.csv", index=False)

# Centro del salar = centro de la mancha más grande del grupo más claro
claro = ndi.binary_opening(GRUPOS == K_GRUPOS, iterations=3)
lab, n = ndi.label(claro)
nucleo = np.zeros_like(claro)
if n:
    k = np.argmax(ndi.sum(claro, lab, range(1, n + 1))) + 1
    nucleo = ndi.binary_fill_holes(lab == k)
if nucleo.sum() * PIX_KM2 >= 5:
    fr, fc = ndi.center_of_mass(nucleo)
    CX, CY = float(xc[int(round(fc))]), float(yc[int(round(fr))])
    lonc, latc = Transformer.from_crs(EPSG, 4326, always_xy=True).transform(CX, CY)
    SITIOS = {"Centro del salar (calculado)": (round(lonc, 4), round(latc, 4)), **SITIOS}
    print(f"Núcleo claro del salar: {nucleo.sum() * PIX_KM2:.0f} km² | centro calculado: {lonc:.4f}, {latc:.4f}")
else:
    CX, CY = Transformer.from_crs(4326, EPSG, always_xy=True).transform((BBOX[0] + BBOX[2]) / 2, (BBOX[1] + BBOX[3]) / 2)
    print("No se encontró el núcleo del salar; uso el centro de la caja")

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(GRUPOS, cmap="viridis", extent=EXT, interpolation="nearest")
ax.set_title(f"Grupos automáticos ({etq[-1]}): 1 = más oscuro"); marcar_sitios(ax)
fig.colorbar(im, ax=ax, ticks=range(1, K_GRUPOS + 1))
plt.savefig(SALIDA / "grupos.png", dpi=150, bbox_inches="tight"); plt.show()

## 8) Cambio fecha a fecha (capa principal)

Para cada fecha se compara la señal con la de la **primera fecha**. Se cuentan los km² que **se aclararon** o **se
oscurecieron** más de UMBRAL_CAMBIO_DB. Es la medida más robusta: no depende de adivinar qué es agua.

In [ ]:
def limpiar(m):
    # Saca puntitos sueltos y rellena huecos chicos
    return ndi.binary_closing(ndi.binary_opening(m, iterations=2), iterations=2)

D = HH - HH[0]                                         # diferencia de cada fecha contra la primera (dB)
ACLARO = np.stack([limpiar(d > UMBRAL_CAMBIO_DB) for d in D])
OSCURECIO = np.stack([limpiar(d < -UMBRAL_CAMBIO_DB) for d in D])
evol = pd.DataFrame({"fecha": etq, "se_aclaro_km2": (ACLARO.sum(axis=(1, 2)) * PIX_KM2).round(2),
                     "se_oscurecio_km2": (OSCURECIO.sum(axis=(1, 2)) * PIX_KM2).round(2)})
display(evol); evol.to_csv(SALIDA / "cambio_por_fecha.csv", index=False)

plt.figure(figsize=(7, 3.5))
f = pd.to_datetime(evol.fecha)
plt.plot(f, evol.se_aclaro_km2, "o-", color="tab:blue", label="se aclaró")
plt.plot(f, evol.se_oscurecio_km2, "o-", color="tab:red", label="se oscureció")
plt.ylabel("km²"); plt.title(f"Superficie con cambio > {UMBRAL_CAMBIO_DB:g} dB respecto de {etq[0]}")
plt.gca().set_xticks(f); plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
plt.grid(alpha=.3); plt.legend(); plt.savefig(SALIDA / "cambio_por_fecha.png", dpi=150, bbox_inches="tight"); plt.show()

## 8b) Comparación contra el promedio de todas las fechas

Por si la primera fecha fue anómala (nieve, helada, humedad), cada fecha también se compara contra el **promedio de las
cinco fechas** (promedio en potencia lineal, pasado a dB). La tabla muestra además el **desvío mediano** de cada fecha
respecto del promedio: si una fecha se aparta mucho de las demás en toda la zona, es sospechosa.

In [ ]:
REF_PROM = a_db(np.nanmean(10 ** (HH / 10), axis=0))
DP = HH - REF_PROM
ACL_P = np.stack([limpiar(d > UMBRAL_CAMBIO_DB) for d in DP])
OSC_P = np.stack([limpiar(d < -UMBRAL_CAMBIO_DB) for d in DP])
evol_prom = pd.DataFrame({"fecha": etq,
    "desvio_mediano_dB": [round(float(np.nanmedian(d)), 2) for d in DP],
    "se_aclaro_vs_promedio_km2": (ACL_P.sum(axis=(1, 2)) * PIX_KM2).round(2),
    "se_oscurecio_vs_promedio_km2": (OSC_P.sum(axis=(1, 2)) * PIX_KM2).round(2)})
display(evol_prom); evol_prom.to_csv(SALIDA / "cambio_vs_promedio.csv", index=False)
anom = evol_prom.loc[evol_prom.desvio_mediano_dB.abs().idxmax()]
print(f"Fecha que más se aparta del promedio: {anom.fecha} ({anom.desvio_mediano_dB:+.2f} dB de desvío mediano)")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
f = pd.to_datetime(evol.fecha)
for a, (tit, ca, co) in zip(ax, [(f"contra {etq[0]}", evol.se_aclaro_km2, evol.se_oscurecio_km2),
                                ("contra el promedio de las fechas", evol_prom.se_aclaro_vs_promedio_km2, evol_prom.se_oscurecio_vs_promedio_km2)]):
    a.plot(f, ca, "o-", color="tab:blue", label="se aclaró"); a.plot(f, co, "o-", color="tab:red", label="se oscureció")
    a.set_title(tit); a.set_xticks(f); a.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m")); a.grid(alpha=.3)
ax[0].set_ylabel("km²"); ax[0].legend()
plt.savefig(SALIDA / "cambio_vs_primera_y_promedio.png", dpi=150, bbox_inches="tight"); plt.show()

## 8c) Cambio persistente

Se queda solo con los píxeles que cambiaron **en el mismo sentido** (respecto de la primera fecha) en al menos
PERSIST_MIN de las fechas posteriores, con un umbral más bajo (UMBRAL_PERSIST_DB) porque la repetición ya filtra el
ruido. Lo que aparece en una sola fecha (nieve, helada, viento) queda afuera; un cambio gradual queda adentro.

In [ ]:
n_post = len(etq) - 1
minimo = min(PERSIST_MIN, n_post)
PERS_ACL = limpiar((D[1:] > UMBRAL_PERSIST_DB).sum(axis=0) >= minimo)
PERS_OSC = limpiar((D[1:] < -UMBRAL_PERSIST_DB).sum(axis=0) >= minimo)
print(f"Cambio persistente (> {UMBRAL_PERSIST_DB:g} dB en el mismo sentido en al menos {minimo} de {n_post} fechas): "
      f"se aclaró {PERS_ACL.sum() * PIX_KM2:.2f} km² · se oscureció {PERS_OSC.sum() * PIX_KM2:.2f} km²")
print(f"Comparado con primera vs última: se aclaró {ACLARO[-1].sum() * PIX_KM2:.2f} km² · se oscureció {OSCURECIO[-1].sum() * PIX_KM2:.2f} km²")

fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(HH[-1], cmap="gray", vmin=-30, vmax=0, extent=EXT)
ax.imshow(np.ma.masked_where(~PERS_ACL, PERS_ACL), cmap=ListedColormap(["#00BFFF"]), alpha=.9, extent=EXT)
ax.imshow(np.ma.masked_where(~PERS_OSC, PERS_OSC), cmap=ListedColormap(["#FF2020"]), alpha=.9, extent=EXT)
marcar_sitios(ax)
ax.set_title(f"Cambio persistente ({etq[0]} a {etq[-1]}): celeste se aclaró · rojo se oscureció\n"
             f"(mismo sentido en al menos {minimo} de {n_post} fechas, > {UMBRAL_PERSIST_DB:g} dB)")
plt.savefig(SALIDA / "cambio_persistente.png", dpi=150, bbox_inches="tight"); plt.show()

## 9) Mapa de cambio y zonas (primera contra última fecha)

In [ ]:
CAMBIO = HH[-1] - HH[0]
rumbos = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE", "S", "SSO", "SO", "OSO", "O", "ONO", "NO", "NNO"]
a_geo = Transformer.from_crs(EPSG, 4326, always_xy=True)
TR = from_origin(xc.min() - abs(xc[1] - xc[0]) / 2, yc.max() + abs(yc[1] - yc[0]) / 2, abs(xc[1] - xc[0]), abs(yc[1] - yc[0]))
zonas = []
for tipo, m in [("se aclaró", ACLARO[-1]), ("se oscureció", OSCURECIO[-1])]:
    lab, n = ndi.label(m)
    for geom, val in rio_features.shapes(lab.astype("int32"), mask=lab > 0, transform=TR):
        g = shape(geom); area = g.area / 1e6
        if area < AREA_MIN_KM2:
            continue
        p = g.representative_point(); dx, dy = p.x - CX, p.y - CY
        ang = (np.degrees(np.arctan2(dx, dy)) + 360) % 360
        lon, lat = a_geo.transform(p.x, p.y)
        zonas.append(dict(tipo=tipo, area_km2=round(area, 2), cambio_medio_dB=round(float(np.nanmean(CAMBIO[lab == int(val)])), 1),
                          ubicacion=f"{np.hypot(dx, dy) / 1000:.1f} km al {rumbos[int((ang + 11.25) // 22.5) % 16]} del centro del salar",
                          lon=round(lon, 4), lat=round(lat, 4), geom=g))
Z = pd.DataFrame(zonas)
if len(Z):
    Z = Z.sort_values("area_km2", ascending=False).reset_index(drop=True); Z.index += 1; Z.index.name = "zona"
    display(Z.drop(columns="geom").head(15))
    Z.drop(columns="geom").to_csv(SALIDA / "zonas_de_cambio.csv", encoding="utf-8-sig")
    print(f"{len(Z)} zonas ≥ {AREA_MIN_KM2} km²:", Z.groupby("tipo").area_km2.sum().round(2).to_dict())

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(CAMBIO, cmap="RdBu", vmin=-6, vmax=6, extent=EXT)
for i, r in Z.iterrows():
    for pg in ([r.geom] if r.geom.geom_type == "Polygon" else list(r.geom.geoms)):
        ax.plot(*pg.exterior.xy, color="black", lw=.8)
    if i <= 12:
        p = r.geom.representative_point()
        ax.annotate(str(i), (p.x, p.y), fontsize=9, fontweight="bold", xytext=(4, 4), textcoords="offset points",
                    bbox=dict(boxstyle="round,pad=0.15", fc="white", alpha=.85, lw=0))
marcar_sitios(ax)
_bx, _by = Transformer.from_crs(4326, EPSG, always_xy=True).transform([CAJA_PILETAS[0], CAJA_PILETAS[2]], [CAJA_PILETAS[1], CAJA_PILETAS[3]])
ax.add_patch(Rectangle((min(_bx), min(_by)), abs(_bx[1] - _bx[0]), abs(_by[1] - _by[0]), fill=False, ec="magenta", lw=1.5, ls="--"))
ax.set_title(f"Zonas de cambio de la señal NISAR (HH): {etq[-1]} respecto de {etq[0]}")
fig.colorbar(im, ax=ax, label="después − antes (dB) · azul: se aclaró · rojo: se oscureció", shrink=.8)
plt.savefig(SALIDA / "zonas_de_cambio.png", dpi=150, bbox_inches="tight"); plt.show()

## 9b) Bloque de piletas: promedio de HH por fecha y superficie con umbral fino

Recorta el bloque de piletas (CAJA_PILETAS) y calcula, para cada fecha: el **promedio de HH y HV** adentro (en potencia
lineal, pasado a dB) y la superficie que cambió más de UMBRAL_PILETAS_DB contra la primera fecha y contra el promedio.
Sin limpieza morfológica, para medir la superficie completa. Responde si la señal baja de forma gradual o en vaivén.

In [ ]:
a_utm = Transformer.from_crs(4326, EPSG, always_xy=True)
bx, by = a_utm.transform([CAJA_PILETAS[0], CAJA_PILETAS[2], CAJA_PILETAS[0], CAJA_PILETAS[2]],
                         [CAJA_PILETAS[1], CAJA_PILETAS[1], CAJA_PILETAS[3], CAJA_PILETAS[3]])
cx_i = np.where((xc >= min(bx)) & (xc <= max(bx)))[0]; cy_i = np.where((yc >= min(by)) & (yc <= max(by)))[0]
sy, sx = slice(cy_i.min(), cy_i.max() + 1), slice(cx_i.min(), cx_i.max() + 1)
def prom_db(arr):
    return float(a_db(np.nanmean(10 ** (arr / 10))))
u = UMBRAL_PILETAS_DB
bloque = pd.DataFrame({"fecha": etq,
    "HH_medio_dB": [round(prom_db(HH[i][sy, sx]), 2) for i in range(len(etq))],
    "HV_medio_dB": [round(prom_db(HV[i][sy, sx]), 2) if np.isfinite(HV[i][sy, sx]).any() else np.nan for i in range(len(etq))],
    f"se_oscurecio_{u:g}dB_vs_primera_km2": [round((D[i][sy, sx] < -u).sum() * PIX_KM2, 2) for i in range(len(etq))],
    f"se_aclaro_{u:g}dB_vs_primera_km2": [round((D[i][sy, sx] > u).sum() * PIX_KM2, 2) for i in range(len(etq))],
    f"se_oscurecio_{u:g}dB_vs_promedio_km2": [round((DP[i][sy, sx] < -u).sum() * PIX_KM2, 2) for i in range(len(etq))],
})
area_caja = HH[0][sy, sx].size * PIX_KM2
print(f"Bloque de piletas: {area_caja:.2f} km² ({HH[0][sy, sx].shape[1]} x {HH[0][sy, sx].shape[0]} píxeles)")
display(bloque); bloque.to_csv(SALIDA / "bloque_piletas_por_fecha.csv", index=False)
dif = np.diff(bloque.HH_medio_dB)
print("Variación de HH medio entre fechas consecutivas (dB):", np.round(dif, 2),
      "→ gradual (siempre en el mismo sentido)" if (np.all(dif <= 0) or np.all(dif >= 0)) else "→ en vaivén (cambia de sentido)")

fig, ax1 = plt.subplots(figsize=(8, 3.8))
f = pd.to_datetime(bloque.fecha)
ax1.plot(f, bloque.HH_medio_dB, "o-", color="black", label="HH medio (dB)")
if bloque.HV_medio_dB.notna().any():
    ax1.plot(f, bloque.HV_medio_dB, "o--", color="gray", label="HV medio (dB)")
ax1.set_ylabel("dB"); ax1.set_xticks(f); ax1.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m")); ax1.grid(alpha=.3)
ax2 = ax1.twinx()
ax2.bar(f, bloque[f"se_oscurecio_{u:g}dB_vs_primera_km2"], width=6, color="tab:red", alpha=.35, label=f"se oscureció > {u:g} dB (km²)")
ax2.set_ylabel("km²"); ax1.set_zorder(ax2.get_zorder() + 1); ax1.patch.set_visible(False)
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax1.legend(h1 + h2, l1 + l2, fontsize=8, loc="lower left")
ax1.set_title("Bloque de piletas: señal media y superficie oscurecida por fecha")
plt.savefig(SALIDA / "bloque_piletas_por_fecha.png", dpi=150, bbox_inches="tight"); plt.show()

ext_b = [xc[sx].min(), xc[sx].max(), yc[sy].min(), yc[sy].max()]
fig, ejes = plt.subplots(1, len(etq), figsize=(3.2 * len(etq), 3.2))
for i, axb in enumerate(np.atleast_1d(ejes)):
    axb.imshow(HH[i][sy, sx], cmap="gray", vmin=-30, vmax=0, extent=ext_b)
    m = D[i][sy, sx] < -u
    axb.imshow(np.ma.masked_where(~m, m), cmap=ListedColormap(["#FF2020"]), alpha=.6, extent=ext_b)
    axb.set_title(etq[i], fontsize=9); axb.set_xticks([]); axb.set_yticks([])
plt.suptitle(f"Bloque de piletas: en rojo, se oscureció más de {u:g} dB respecto de {etq[0]}")
plt.savefig(SALIDA / "bloque_piletas_paneles.png", dpi=130, bbox_inches="tight"); plt.show()

## 10) Agua o salmuera (capa secundaria, PROVISORIA)

En la Puna el terreno seco y liso también se ve oscuro en el radar. Por eso esta capa usa un umbral muy bajo y hay que
confirmarla con imagen óptica antes de mostrarla. Mirá el histograma: el agua calma debería formar una cola a la izquierda.

In [ ]:
plt.figure(figsize=(7, 3))
plt.hist(HH[-1][np.isfinite(HH[-1])], bins=150, color="gray")
plt.axvline(UMBRAL_AGUA_DB, color="red", label=f"umbral {UMBRAL_AGUA_DB} dB")
plt.xlabel("HH (dB)"); plt.ylabel("píxeles"); plt.title("Histograma, última fecha"); plt.legend(); plt.show()

AGUA = np.stack([limpiar(h < UMBRAL_AGUA_DB) for h in HH])
areas = pd.DataFrame({"fecha": etq, "area_muy_oscura_km2": (AGUA.sum(axis=(1, 2)) * PIX_KM2).round(2)})
display(areas); areas.to_csv(SALIDA / "area_muy_oscura_por_fecha_PROVISORIO.csv", index=False)

## 11) Deslizador de fechas y GIF

In [ ]:
cuadros = []
for i in range(len(etq)):
    fig, ax = plt.subplots(figsize=(6, 5.5))
    ax.imshow(HH[i], cmap="gray", vmin=-30, vmax=0, extent=EXT)
    if i > 0:
        ax.imshow(np.ma.masked_where(~ACLARO[i], ACLARO[i]), cmap=ListedColormap(["#00BFFF"]), alpha=.85, extent=EXT)
        ax.imshow(np.ma.masked_where(~OSCURECIO[i], OSCURECIO[i]), cmap=ListedColormap(["#FF2020"]), alpha=.85, extent=EXT)
    marcar_sitios(ax, chico=True)
    ax.set_title(f"NISAR HH {etq[i]}\nazul: se aclaró · rojo: se oscureció (vs {etq[0]})", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    fig.canvas.draw(); cuadros.append(np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()); plt.close(fig)
imageio.mimsave(SALIDA / "serie_salar.gif", cuadros, duration=1.2, loop=0)
print("GIF guardado:", SALIDA / "serie_salar.gif")

try:
    from ipywidgets import interact, IntSlider
    @interact(i=IntSlider(min=0, max=len(etq) - 1, step=1, value=len(etq) - 1, description="Fecha"))
    def ver(i):
        plt.figure(figsize=(7, 6.5)); plt.imshow(cuadros[i]); plt.axis("off"); plt.show()
except Exception as e:
    print("Sin deslizador en este entorno:", e)

## 12) Exportar y descargar

In [ ]:
def a_tif(nombre, arr, dtype="float32", nodata=np.nan):
    tr = from_origin(xc.min() - abs(xc[1] - xc[0]) / 2, yc.max() + abs(yc[1] - yc[0]) / 2, abs(xc[1] - xc[0]), abs(yc[1] - yc[0]))
    extra = {"predictor": 3, "zlevel": 9} if dtype == "float32" else {"predictor": 2}
    with rasterio.open(SALIDA / nombre, "w", driver="GTiff", height=arr.shape[0], width=arr.shape[1], count=1,
                       dtype=dtype, crs=f"EPSG:{EPSG}", transform=tr, nodata=nodata, compress="deflate", **extra) as ds:
        # los rásteres en dB se redondean a 0,1 dB: pesan la mitad y no cambia nada del análisis
        ds.write(np.nan_to_num(arr, nan=nodata).astype(dtype) if dtype != "float32" else np.round(arr, 1).astype(dtype), 1)

a_tif("cambio_HH_dB.tif", CAMBIO)
a_tif("grupos.tif", GRUPOS, "uint8", 0)
a_tif("cambio_persistente.tif", (PERS_ACL * 1 + PERS_OSC * 2).astype("uint8"), "uint8", 0)   # 1 = se aclaró, 2 = se oscureció
a_tif("cambio_vs_promedio_ultima_dB.tif", DP[-1])
for i, e in enumerate(etq):
    a_tif(f"muy_oscuro_PROVISORIO_{e}.tif", AGUA[i].astype("uint8"), "uint8", 255)
    if EXPORTAR_HH:
        a_tif(f"HH_dB_{e}.tif", HH[i])
    if EXPORTAR_CAMBIO_FECHAS and i > 0:
        a_tif(f"cambio_HH_dB_{e}.tif", D[i])   # cambio de esta fecha contra la primera (dB)

if len(Z):
    to_geo = lambda x, y, z=None: a_geo.transform(x, y)
    from shapely.ops import transform as s_transform
    fc = {"type": "FeatureCollection", "features": []}
    kml = ['<?xml version="1.0" encoding="UTF-8"?><kml xmlns="http://www.opengis.net/kml/2.2"><Document>',
           f'<name>Zonas de cambio NISAR {etq[0]} a {etq[-1]}</name>',
           '<Style id="a"><LineStyle><color>ffff8800</color><width>2</width></LineStyle><PolyStyle><color>66ff8800</color></PolyStyle></Style>',
           '<Style id="o"><LineStyle><color>ff0000dd</color><width>2</width></LineStyle><PolyStyle><color>660000dd</color></PolyStyle></Style>']
    for i, r in Z.iterrows():
        g4 = s_transform(to_geo, r.geom.simplify(20))
        fc["features"].append({"type": "Feature", "geometry": mapping(g4), "properties": {"zona": int(i), "tipo": r.tipo,
                               "area_km2": float(r.area_km2), "cambio_medio_dB": float(r.cambio_medio_dB), "ubicacion": r.ubicacion}})
        for pg in ([g4] if g4.geom_type == "Polygon" else list(g4.geoms)):
            co = " ".join(f"{x:.6f},{y:.6f},0" for x, y in pg.exterior.coords)
            kml.append(f'<Placemark><name>Zona {i}</name><description>{r.tipo}; {r.area_km2} km²; {r.cambio_medio_dB} dB; {r.ubicacion}</description>'
                       f'<styleUrl>#{"a" if r.tipo == "se aclaró" else "o"}</styleUrl><Polygon><outerBoundaryIs><LinearRing><coordinates>{co}</coordinates></LinearRing></outerBoundaryIs></Polygon></Placemark>')
    for nombre, (lo, la) in SITIOS.items():
        kml.append(f"<Placemark><name>{nombre}</name><Point><coordinates>{lo},{la},0</coordinates></Point></Placemark>")
    kml.append("</Document></kml>")
    (SALIDA / "zonas_de_cambio.geojson").write_text(json.dumps(fc, ensure_ascii=False), encoding="utf-8")
    with zipfile.ZipFile(SALIDA / "zonas_de_cambio.kmz", "w", zipfile.ZIP_DEFLATED) as k:
        k.writestr("doc.kml", "\n".join(kml))

(SALIDA / "LEEME.txt").write_text(
    "NISAR GCOV (NASA/JPL-ISRO), ASF DAAC. Órbita %s, frame %s. Fechas: %s.\n"
    "Filtro %dx%d. Cambio = HH(fecha) - HH(primera); zonas: |cambio| > %.1f dB y area >= %.2f km2.\n"
    "Capa 'agua': HH < %.1f dB, PROVISORIA (el terreno seco y liso tambien se ve oscuro).\n"
    "Grupos: k-means con HH y HV, sin validar. Describe cambios de superficie; no atribuye causas.\n"
    "NASA no respalda este trabajo.\n" % (orbita, frame, ", ".join(etq), FILTRO_PX, FILTRO_PX, UMBRAL_CAMBIO_DB, AREA_MIN_KM2, UMBRAL_AGUA_DB), encoding="utf-8")

zip_path = DATA_DIR / "resultados_hombre_muerto.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in SALIDA.iterdir():
        z.write(p, p.name)
print("ZIP:", zip_path, f"({zip_path.stat().st_size / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Fuera de Colab: el ZIP queda en la carpeta indicada.")

## Límites (para copiar en la presentación)

- Cinco meses de datos (invierno a primavera) no separan lo estacional de una tendencia.
- "Oscuro" no es siempre agua: el agua con viento se ve clara y algunas superficies lisas se ven oscuras. Confirmar con imagen óptica.
- El umbral de agua y los grupos automáticos no están validados en campo.
- Hay un hueco de datos de NISAR del 27/07 al 10/08/2026.